# Amazon Nova — picking a tier, and proving the small one is enough

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Nova is Amazon's own model family and the clearest size ladder on Bedrock. It is
`bedrock-runtime` only: there is no `bedrock-mantle` path, so Converse is the
API.

| Model | Input | Tier |
|---|---|---|
| `nova-micro` | text | cheapest, text only |
| `nova-lite` | text, image, video | low cost, multimodal |
| `nova-pro` | text, image, video | higher quality, multimodal |
| `nova-2-lite` | text, image, video | generation 2, multimodal; **no bare-ID access** (see section 5) |
| `nova-premier` | — | **superseded**: gone from `ListFoundationModels`, profile still listed, calls refused (see section 4) |

The interesting question with a ladder is never "which is best" — it is "what is
the cheapest tier that still passes". This notebook answers that empirically
rather than by reputation. `nova-2-lite` is a second generation rather than a
fourth rung, so it is measured alongside the ladder on the same
tasks instead of being slotted into it.

It also addresses differently, and which models take a bare model ID is keyed on
the Region as well as the model. `nova-2-lite` is `INFERENCE_PROFILE`-only in both
Regions measured below; the generation-1 rungs are not consistent across Regions.
Section 5 measures two Regions against the catalogue flag rather than asking you to
remember a rule.

Nova also accepts **video** input on lite and pro. Video is out of scope for this
collection, so the vision cells use images; the same content-block shape applies.


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `slide_jpeg` | JPEG bytes of a slide from a public AWS talk, so vision cells have a known answer |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `list_models` | the `bedrock-mantle` model catalogue for a Region; raises if it cannot be read |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |
| `control_client` | a boto3 `bedrock` client (model and profile catalogues) |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    SLIDE_CALLOUTS,
    SLIDE_TITLE,
    converse,
    converse_tool_uses,
    endpoints_for,
    keyword_recall,
    list_models,
    resolve_runtime_id,
    runtime_models,
    slide_jpeg,
)

REGION = "us-east-1"
MICRO = "amazon.nova-micro-v1"
LITE = "amazon.nova-lite-v1"
PRO = "amazon.nova-pro-v1"
NOVA2_LITE = "amazon.nova-2-lite-v1"

# LADDER is the generation-1 size ladder, which is what sections 1-3 compare.
# NOVA2_LITE is a different generation, so it rides along in MODELS rather than
# being treated as a fourth rung.
LADDER = [MICRO, LITE, PRO]
MODELS = LADDER + [NOVA2_LITE]

catalogue = runtime_models(REGION)
print(f"{'model':<26} {'input':<22} {'inference types'}")
print("-" * 76)
for model in MODELS:
    entry = catalogue[model]
    print(f"{model:<26} {','.join(sorted(entry['in'])):<22} "
          f"{','.join(sorted(entry['infer']))}")

print()
print("endpoints:", {m.split(".")[-1]: endpoints_for(m, REGION) for m in [MICRO]})

# Derive the runtime-only verdict rather than asserting it. endpoints_for() reports
# mantle=False both when a model is absent from that catalogue and when the
# catalogue could not be read at all, so read the catalogue here and let the two
# cases print differently. The unconditional line this replaces printed "Nova is a
# runtime-only family" even on a host whose role cannot reach bedrock-mantle, where
# the run had measured nothing.
try:
    amazon_on_mantle = [m for m in list_models(REGION) if m.startswith("amazon.")]
except Exception as exc:
    print(f"=> bedrock-mantle catalogue unreadable ({type(exc).__name__}): whether")
    print("   Nova is runtime-only is UNVERIFIED in this run, not confirmed.")
else:
    print(f"amazon.* on bedrock-mantle in {REGION}: {amazon_on_mantle or 'none'}")
    if amazon_on_mantle:
        print("=> an Amazon model is on bedrock-mantle: the runtime-only premise of")
        print("   this notebook is stale, and the sections below need revisiting.")
    else:
        print("=> nothing under amazon.* on bedrock-mantle: Nova is runtime-only.")


model                      input                  inference types
----------------------------------------------------------------------------
amazon.nova-micro-v1       TEXT                   INFERENCE_PROFILE,ON_DEMAND,PROVISIONED
amazon.nova-lite-v1        IMAGE,TEXT,VIDEO       INFERENCE_PROFILE,ON_DEMAND,PROVISIONED
amazon.nova-pro-v1         IMAGE,TEXT,VIDEO       INFERENCE_PROFILE,ON_DEMAND,PROVISIONED
amazon.nova-2-lite-v1      IMAGE,TEXT,VIDEO       INFERENCE_PROFILE,PROVISIONED



endpoints: {'nova-micro-v1': {'mantle': False, 'runtime': True}}


amazon.* on bedrock-mantle in us-east-1: none
=> nothing under amazon.* on bedrock-mantle: Nova is runtime-only.


## 1. The same task at each tier, and at generation 2

A single easy prompt tells you nothing — every tier passes. Use a task with a
checkable answer and enough structure that a weaker model can visibly fail.

Below: extract three fields as JSON. The check is mechanical, so "did it pass"
is not a judgement call.


In [2]:
import json as jsonlib

PROMPT = (
    "Extract to JSON with keys name, city, years. "
    "Reply with JSON only, no prose.\n\n"
    "Priya has been an engineer in Singapore for eleven years."
)
EXPECTED = {"name": "Priya", "city": "Singapore", "years": 11}


def grade(raw: str) -> str:
    """Did the model return the three fields with the right values?"""
    text = (raw or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        text = text[4:] if text.startswith("json") else text
    try:
        got = jsonlib.loads(text.strip())
    except Exception:
        return "unparseable"
    hits = sum(
        1
        for key, want in EXPECTED.items()
        if str(got.get(key, "")).lower() == str(want).lower()
    )
    return f"{hits}/3 fields correct"


print(f"{'model':<26} {'tokens':>7}  {'verdict':<22} answer")
print("-" * 92)
for model in MODELS:
    text, response = converse(
        model,
        [{"role": "user", "content": [{"text": PROMPT}]}],
        max_tokens=200,
        temperature=0.0,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} {'-':>7}  ERROR {error[:40]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    print(f"{model:<26} {total:>7}  {grade(text):<22} "
          f"{text.strip()[:34].replace(chr(10), ' ')}")


model                       tokens  verdict                answer
--------------------------------------------------------------------------------------------


amazon.nova-micro-v1            57  3/3 fields correct     {   "name": "Priya",   "city": "Si


amazon.nova-lite-v1             61  3/3 fields correct     ```json {     "name": "Priya",    


amazon.nova-pro-v1              61  3/3 fields correct     ```json {   "name": "Priya",   "ci


amazon.nova-2-lite-v1          108  3/3 fields correct     ```json {   "name": "Priya",   "ci


## 2. Vision on the three multimodal models

`nova-micro` is text-only, so sending it an image is a design error rather than a
quality question. The three multimodal models get the same generated image with a
known answer, so "did it look" is verifiable.


In [3]:
jpeg = slide_jpeg()
QUESTION = "Read this slide. Give its title, then quote the three green callout lines."
print(f"slide: {len(jpeg)} bytes of JPEG; title is {SLIDE_TITLE!r}\n")

for model in (LITE, PRO, NOVA2_LITE):
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [
                    {"image": {"format": "jpeg", "source": {"bytes": jpeg}}},
                    {"text": QUESTION},
                ],
            }
        ],
        max_tokens=220,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:60]}")
        continue
    hits, total = keyword_recall(text, SLIDE_CALLOUTS)
    title_seen = SLIDE_TITLE.lower() in (text or "").lower()
    print(f"{model:<26} callouts {hits}/{total}  title={title_seen}")
    print(f"{'':<26} {' '.join(text.split())[:90]!r}")

# And the design error, so you recognise it.
text, response = converse(
    MICRO,
    [
        {
            "role": "user",
            "content": [
                {"image": {"format": "jpeg", "source": {"bytes": jpeg}}},
                {"text": QUESTION},
            ],
        }
    ],
    max_tokens=40,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
print(f"\n{MICRO} (text-only) with an image:")
print("   ", error[:130] if error else f"unexpectedly accepted: {text.strip()[:60]}")

slide: 32687 bytes of JPEG; title is 'Ingestion from database'



amazon.nova-lite-v1        callouts 3/3  title=True
                           'Title: Ingestion from database Callout 1: Pay according to job duration Callout 2: Lower s'


amazon.nova-pro-v1         callouts 3/3  title=True
                           'The slide is titled "Ingestion from database." The three green callout lines read: - Pay a'


amazon.nova-2-lite-v1      callouts 3/3  title=True
                           '### Title of the Slide: **Ingestion from database** ### Three Green Callout Lines: 1. **✓ '

amazon.nova-micro-v1 (text-only) with an image:
    This model doesn't support the image content block that you provided. Update the content block and try again.


## 3. Tool use across the family

Tool support is not a given at the cheapest tier, so check it rather than assume.
The assertion here is on the *arguments*, not on whether a call happened — a tool
call with wrong operands still reports `stopReason: tool_use`.


In [4]:
TOOLS = [
    {
        "toolSpec": {
            "name": "convert_currency",
            "description": "Convert an amount between two currencies",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "amount": {"type": "number"},
                        "from": {"type": "string"},
                        "to": {"type": "string"},
                    },
                    "required": ["amount", "from", "to"],
                }
            },
        }
    }
]

print(f"{'model':<26} {'stop':<12} tool call")
print("-" * 78)
for model in MODELS:
    text, response = converse(
        model,
        [
            {
                "role": "user",
                "content": [{"text": "Convert 250 SGD to JPY. Use the tool."}],
            }
        ],
        max_tokens=400,
        tools=TOOLS,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:44]}")
        continue
    uses = converse_tool_uses(response)
    if not uses:
        print(f"{model:<26} {response.get('stopReason'):<12} (no tool call)")
        continue
    args = uses[0]["input"]
    ok = (
        str(args.get("amount")) in {"250", "250.0"}
        and str(args.get("from", "")).upper() == "SGD"
        and str(args.get("to", "")).upper() == "JPY"
    )
    print(f"{model:<26} {response.get('stopReason'):<12} "
          f"{args}  {'correct' if ok else 'ARGS WRONG'}")


model                      stop         tool call
------------------------------------------------------------------------------


amazon.nova-micro-v1       tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


amazon.nova-lite-v1        tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


amazon.nova-pro-v1         tool_use     {'amount': 250, 'from': 'SGD', 'to': 'JPY'}  correct


amazon.nova-2-lite-v1      tool_use     {'amount': 250.0, 'from': 'SGD', 'to': 'JPY'}  correct


## 4. `nova-premier` is superseded, and no single catalogue says so

A superseded model does not leave every surface at once. `nova-premier` is no longer
in `ListFoundationModels`, and its `us.` inference profile is still listed: a caller
who checks the profile list concludes the model is there, and the call fails anyway.

That is why "read the catalogue" is not sufficient on its own. Each list answers
about its own surface, and none of them answers "may I invoke this today". The cell
below reads both lists, then calls the model, and derives its verdict from the call.

The refusal wording is not a stable thing to match on — it changed under this
notebook. An earlier committed run of this cell recorded

    Model is marked by provider as Legacy and you have not been actively using
    the model in the last 30 days. Please upgrade to an active model

for the same `ResourceNotFoundException` the run below records differently. Match
the exception type, not the sentence.


In [ ]:
from bedrock import control_client, inference_profiles, runtime_client

PREMIER = "amazon.nova-premier-v1"
PREMIER_ID = f"{PREMIER}:0"

# Surface 1: the model catalogue read in the setup cell.
entry = catalogue.get(PREMIER)
in_catalogue = entry is not None
print(f"in ListFoundationModels  : {in_catalogue}")
print(f"  inference types        : {sorted(entry['infer']) if entry else '-'}")

# Surface 2: the profile list, which is where this model still is. The two lists
# disagree, and the disagreement is the lesson. inference_profiles() returns an
# empty set both when a Region has no profiles and when the caller lacks
# bedrock:ListInferenceProfiles, so empty is "not measured" here, not "absent".
profiles = inference_profiles(REGION)
profile_id = f"us.{PREMIER_ID}"
profile_listed = None
if profiles:
    profile_listed = profile_id in profiles
    print(f"in ListInferenceProfiles : {profile_listed}  ({profile_id})")
else:
    print("in ListInferenceProfiles : not measured (empty or unreadable)")

# Surface 3: the entitlement check. It needs bedrock:GetFoundationModelAvailability,
# which is not in the IAM set the rest of this collection asks for, so a role
# without it must print that and carry on rather than stopping the notebook here.
try:
    availability = control_client(REGION).get_foundation_model_availability(
        modelId=PREMIER_ID
    )
    print("  authorizationStatus    :", availability.get("authorizationStatus"))
    print("  entitlement            :", availability.get("entitlementAvailability"))
except Exception as exc:
    print(f"entitlement check        : not measured ({type(exc).__name__}),")
    print("                           needs bedrock:GetFoundationModelAvailability")

# The only surface that answers "may I invoke this": the call itself.
print("\ncalling it, through whichever form resolve_runtime_id() picks:")
called_ok = False
try:
    runtime_client(REGION).converse(
        modelId=resolve_runtime_id(PREMIER, REGION),
        messages=[{"role": "user", "content": [{"text": "hi"}]}],
        inferenceConfig={"maxTokens": 12},
    )
    called_ok = True
    print("    accepted")
except Exception as exc:
    # Read the code and message out of the botocore error rather than slicing
    # str(exc): the slice that was here cut the sentence mid-word, and the point of
    # printing it at all is that the sentence is worth reading once and then not
    # matching on.
    error = getattr(exc, "response", {}).get("Error", {})
    print(f"    {error.get('Code', type(exc).__name__)}")
    print(f"    {error.get('Message', str(exc))}")

# Verdict from the three readings above, not from the heading of this section.
print()
if called_ok:
    print(f"=> {PREMIER} accepted a call in {REGION}: this section is stale.")
elif in_catalogue or profile_listed:
    listed_on = [
        name
        for name, present in (
            ("ListFoundationModels", in_catalogue),
            ("ListInferenceProfiles", profile_listed),
        )
        if present
    ]
    print(f"=> refused, and still listed by {' and '.join(listed_on)}:")
    print("   presence on a catalogue is not permission to invoke.")
else:
    print("=> refused, and absent from every list read here: in this Region there")
    print("   is no longer a catalogue entry to be misled by.")


## 5. Bare-ID access follows the `ON_DEMAND` flag, and the flag is per Region

Section 4 showed that being listed is not permission. This section asks a narrower
question: when the catalogue says a model supports `ON_DEMAND`, does a **bare model
ID** actually work, and when it says `INFERENCE_PROFILE` only, is the bare ID
actually refused?

That matters because it is the difference between working and 400 for the same code:

    Invocation of model ID amazon.nova-2-lite-v1:0 with on-demand throughput
    isn’t supported. Retry your request with the ID or ARN of an inference
    profile that contains this model.

It is measured in **two Regions**, because a single Region cannot tell a rule about
the model from a rule about the Region, and this is a case where the two differ. The
`ON_DEMAND` flag is carried per Region by `ListFoundationModels`, so the same
generation-1 model can take a bare ID in one Region and refuse it in another.

`resolve_runtime_id()` is what the rest of this collection uses to avoid caring; it
prefers the geo-prefixed profile whenever one exists, which is required for an
`INFERENCE_PROFILE`-only model and strictly better for the rest. The cell below calls
each model **twice per Region** — once with the bare catalogue ID and once with the
resolved ID — and reports three things. Read them in this order:

1. whether the flag agreed with the bare-ID result, per Region. If that is not
   4/4 the flag is not a reliable predictor there, and the table says which model
   broke it.
2. whether every resolved ID was accepted, which is the claim `resolve_runtime_id()`
   makes for itself.
3. whether the set of models that took a bare ID is the same in both Regions. If it
   is not, no single sentence about "the generation-1 models" is true, and the
   Region has to be part of any rule you write down.


In [ ]:
HELLO = [{"role": "user", "content": [{"text": "Reply with the single word: ok"}]}]

# Probe REGION first, then the other of the two Regions this notebook compares, so
# that changing REGION above still gives a two-Region comparison rather than
# measuring one Region twice.
REGIONS = [REGION] + [r for r in ("us-east-1", "us-west-2") if r != REGION][:1]


def call_verdict(model_id: str, region: str) -> tuple[bool, str, str]:
    """Call one exact model ID with no resolution. Returns (ok, label, message).

    resolve=False is the point of this cell: resolve_runtime_id() would rewrite a
    bare ID into a profile ID and hide the very difference being measured.
    """
    _, response = converse(
        model_id, HELLO, max_tokens=16, region=region, resolve=False
    )
    error = response.get("error") or {}
    if not error:
        return True, "accepted", ""
    # Label by error code, not by a slice of the message: the message contains the
    # model ID, and model IDs contain dots, so splitting on one truncates mid-word.
    return False, f"refused: {error.get('code', 'error')}", error.get("message", "")


bare_ok_by_region: dict[str, set[str]] = {}
# Refusal messages differ only by the model ID they name, so group by the message
# with that ID elided. Four near-identical paragraphs hide whether anything was
# actually different; one shape per group shows it.
refusal_shapes: dict[str, list[str]] = {}

for region in REGIONS:
    try:
        region_catalogue = runtime_models(region)
    except Exception as exc:
        # No catalogue means no ON_DEMAND flag to test the behaviour against, so
        # this Region contributes nothing rather than contributing a guess.
        print(f"{region}: catalogue unreadable ({type(exc).__name__}), not measured\n")
        continue
    print(f"=== {region}")
    print(f"{'model':<26} {'ON_DEMAND':<10} {'bare ID':<32} resolved ID")
    print("-" * 104)
    agreed = 0
    resolved_all_ok = True
    took_bare: set[str] = set()
    for model in MODELS:
        entry = region_catalogue[model]
        on_demand = "ON_DEMAND" in entry["infer"]
        bare_ok, bare, message = call_verdict(entry["id"], region)
        resolved_id = resolve_runtime_id(model, region)
        resolved_ok, resolved, _ = call_verdict(resolved_id, region)
        # The claim under test: does the ON_DEMAND flag predict the bare-ID result?
        if on_demand == bare_ok:
            agreed += 1
        if bare_ok:
            took_bare.add(model)
        resolved_all_ok = resolved_all_ok and resolved_ok
        if message:
            shape = message.replace(entry["id"], "<model-id>")
            refusal_shapes.setdefault(shape, []).append(f"{region} {entry['id']}")
        print(f"{model:<26} {str(on_demand):<10} {bare:<32} {resolved_id}")
        print(f"{'':<26} {'':<10} {'':<32} {resolved}")
    bare_ok_by_region[region] = took_bare
    print()
    print(f"ON_DEMAND flag agreed with the bare-ID result: {agreed}/{len(MODELS)}")
    print(f"every resolved ID accepted: {resolved_all_ok}")
    print()

# Is bare-ID access a property of the model, or of the model and the Region? Compare
# the sets rather than asserting either answer.
if len(bare_ok_by_region) < 2:
    print("=> only one Region measured, so this run cannot say whether bare-ID")
    print("   access is keyed on the Region.")
else:
    sets = list(bare_ok_by_region.values())
    if all(s == sets[0] for s in sets):
        names = ", ".join(bare_ok_by_region)
        print(f"=> the same models took a bare ID in {names}: nothing here keys on")
        print("   the Region.")
    else:
        print("=> bare-ID access differs by Region, so the model alone does not")
        print("   decide it:")
        for region, took in bare_ok_by_region.items():
            listed = ", ".join(sorted(took)) or "none"
            print(f"     {region:<12} bare ID accepted for: {listed}")

for shape, keys in refusal_shapes.items():
    print(f"\n{len(keys)} bare ID(s) refused with this message, model ID elided:")
    for key in keys:
        print(f"    {key}")
    print(f"    {shape}")


## Takeaways

- **Nova is `bedrock-runtime` only.** No `bedrock-mantle` path, so no bearer token
  and no OpenAI-shaped option.
- **Pick the tier with a graded task, not a vibe.** Section 1 gives a mechanical
  pass mark; if `nova-micro` scores 3/3 on your real task, the higher tiers are
  spend without return.
- **`nova-micro` is text-only.** Sending it an image is a design error, not a
  quality trade-off.
- **Assert tool arguments.** `stopReason: tool_use` only says a call was made, not
  that it was right.
- **Being listed is not permission, and the lists disagree with each other.**
  `nova-premier` has left `ListFoundationModels` while its `us.` inference profile
  is still listed, and it refuses either way. Sweep your intended model list with
  one cheap call before you design around it, and match the exception type rather
  than the refusal wording, which has already changed once here.
- **Whether a bare model ID works is keyed on the Region, not just the model.**
  `nova-2-lite` is `INFERENCE_PROFILE`-only in both Regions measured, but the
  generation-1 rungs are not, so a bare ID that works where you developed can be
  refused with a `ValidationException` where you deploy. Section 5 measures the bare
  and resolved forms in two Regions; pass model IDs through `resolve_runtime_id()`
  rather than hardcoding either form.
